## Data Ingestion

In [57]:
### Document Datastructure

from langchain_core.documents import Document

In [58]:
doc = Document(
    page_content = "this is the main content I am using to create RAG",
    metadata = {
        "source":"example.txt",
        "pages":1,
        "author":"Mathura Santhalingam",
        "date_created":"20/08/2026"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Mathura Santhalingam', 'date_created': '20/08/2026'}, page_content='this is the main content I am using to create RAG')

In [59]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)


In [60]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
     
    """
}
for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print(" Sample text files created.")

 Sample text files created.


In [61]:
### Text Loader (how to read the text above using TextLoader)
from langchain.document_loaders import TextLoader
# OR : from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")

# loader is the act of reading, it will give an object of Textloader
document=loader.load() # to get the content inside it
print(document)


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [62]:
### Directory Loader(how to read the text above using DirectoryLoader, i want to read the text when i have all the important files in my directory)
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader = DirectoryLoader(
    "../data/text_files", 
    glob="**/*.txt", ## pattern to match files on the files
    loader_cls= TextLoader, ##loader class to use, which file we plan to load, if pdf, write pdf instead of TextLoaders
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'),
 Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervise

In [63]:
### for PDF
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader = DirectoryLoader(
    "../data/pdf/", 
    glob="**/*.pdf",
    loader_cls= PyMuPDFLoader,
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'macOS Version 26.5.2 (assemblage 25F84) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20260819230533Z00'00'", 'source': '../data/pdf/def.pdf', 'file_path': '../data/pdf/def.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'def', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260819230533Z00'00'", 'trapped': '', 'modDate': "D:20260819230533Z00'00'", 'creationDate': "D:20260819230533Z00'00'", 'page': 0}, page_content="Dans une architecture RAG (Retrieval-Augmented Generation), l'ingestion des données constitue la \npremière étape critique pour enrichir le contexte d'un LLM avec des sources externes. \nLe rôle des loaders \nLes document loaders servent de passerelle entre les formats d'origine et le pipeline de traitement. \nQu'il s'agisse de fichiers locaux (PDF, Markdown, CSV), de pages web ou de bases de données \n(Notion, SQL), le chargeur extrait le contenu brut, nettoie les données non pertinentes et standardise \nle f

In [64]:
type(pdf_documents[0]) ###as the above is a list of documents

langchain_core.documents.base.Document

### Chunking

In [65]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [66]:
chunks=split_documents(pdf_documents)
chunks

Split 41 documents into 214 chunks

Example chunk:
Content: Dans une architecture RAG (Retrieval-Augmented Generation), l'ingestion des données constitue la 
première étape critique pour enrichir le contexte d'un LLM avec des sources externes. 
Le rôle des loa...
Metadata: {'producer': 'macOS Version 26.5.2 (assemblage 25F84) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20260819230533Z00'00'", 'source': '../data/pdf/def.pdf', 'file_path': '../data/pdf/def.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'def', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260819230533Z00'00'", 'trapped': '', 'modDate': "D:20260819230533Z00'00'", 'creationDate': "D:20260819230533Z00'00'", 'page': 0}


[Document(metadata={'producer': 'macOS Version 26.5.2 (assemblage 25F84) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20260819230533Z00'00'", 'source': '../data/pdf/def.pdf', 'file_path': '../data/pdf/def.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'def', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260819230533Z00'00'", 'trapped': '', 'modDate': "D:20260819230533Z00'00'", 'creationDate': "D:20260819230533Z00'00'", 'page': 0}, page_content="Dans une architecture RAG (Retrieval-Augmented Generation), l'ingestion des données constitue la \npremière étape critique pour enrichir le contexte d'un LLM avec des sources externes. \nLe rôle des loaders \nLes document loaders servent de passerelle entre les formats d'origine et le pipeline de traitement. \nQu'il s'agisse de fichiers locaux (PDF, Markdown, CSV), de pages web ou de bases de données \n(Notion, SQL), le chargeur extrait le contenu brut, nettoie les données non pertinentes et standardise \nle f

### Embedding and VectorStoreDB

In [67]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid # Universally Unique Identifier
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [68]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args :
            model_name : HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    # def get_embedding_dimension(self) -> int:
    #     """Get the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded")
    #     return self.model.get_sentence_embedding_dimension()
    
# initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


### VectorStore

In [69]:
class VectorStore:
    """Manages document embedding in a ChromaDB vector store"""

    def __init__(self, collection_name:str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Intialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection (where will be stored the vectors in the VDB)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Colelction: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

        #  now that we have created the collection, we need to add the documents
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store=VectorStore() # it will show 0 since we didn't add any document in the collection
vector_store

Vector store initialized. Colelction: pdf_documents
Existing documents in collection: 144


In [70]:
chunks

[Document(metadata={'producer': 'macOS Version 26.5.2 (assemblage 25F84) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20260819230533Z00'00'", 'source': '../data/pdf/def.pdf', 'file_path': '../data/pdf/def.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'def', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260819230533Z00'00'", 'trapped': '', 'modDate': "D:20260819230533Z00'00'", 'creationDate': "D:20260819230533Z00'00'", 'page': 0}, page_content="Dans une architecture RAG (Retrieval-Augmented Generation), l'ingestion des données constitue la \npremière étape critique pour enrichir le contexte d'un LLM avec des sources externes. \nLe rôle des loaders \nLes document loaders servent de passerelle entre les formats d'origine et le pipeline de traitement. \nQu'il s'agisse de fichiers locaux (PDF, Markdown, CSV), de pages web ou de bases de données \n(Notion, SQL), le chargeur extrait le contenu brut, nettoie les données non pertinentes et standardise \nle f

In [71]:
# Convert the text to embeddings
texts=[doc.page_content for doc in chunks]
texts 
# extract all the text from a particular chunk and we will generate an embedding
# and we will pass it to embedding_manager

# Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

# Store in the VectorDataBase
vector_store.add_documents(chunks, embeddings)


Generating embeddings for 214 texts...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches: 100%|██████████| 7/7 [00:03<00:00,  1.93it/s]


Generated embeddings with shape: (214, 384)
Adding 214 documents to vector store...
Successfully added 214 documents to vector store
Total documents in collection: 358


### Retriever Pipeline from VectorStore

In [72]:
class RAGRetriever:
    """Handles query-baser retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k:int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold} ")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                # we zip them and compare distance
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        # we add it inside the context documents which is created in retrieved_docs 
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print ("No documents found")
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever

In [73]:
# example with a query
rag_retriever.retrieve("What is a chain of verification")

Retrieving documents for query: 'What is a chain of verification'
Top K: 5, Score threshold: 0.0 
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 82.43it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_bbc8aad3_14',
  'content': 'Our overall process, which we call Chain-of-Verification (CoVe), thus performs four core steps:\n1. Generate Baseline Response: Given a query, generate the response using the LLM.\n2. Plan Verifications: Given both query and baseline response, generate a list of verification\nquestions that could help to self-analyze if there are any mistakes in the original response.\n3. Execute Verifications: Answer each verification question in turn, and hence check the answer\nagainst the original response to check for inconsistencies or mistakes.\n4. Generate Final Verified Response: Given the discovered inconsistencies (if any), generate a\nrevised response incorporating the verification results.\nEach of these steps is performed by prompting the same LLM in different ways to obtain the desired\nresponse. While steps (1), (2) and (4) all can be invoked with a single prompt, we investigate\nvariations of step (3) including joint, 2-step and factored version

### Integration VectorDB Context Pipeline with LLM Output

In [74]:
# Simple RAG Pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set the GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="qwen/qwen3.8-27b",temperature=0.1,max_tokens=1024)

## 2.Simple RAG function : retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else "" # whatever info i get from res, just combine everything and put it inside this
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""

    response=llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [75]:
answer=rag_simple("What is chain of verification ?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is chain of verification ?'
Top K: 3, Score threshold: 0.0 
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 90.68it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Chain-of-Verification (CoVe) is a process that performs four core steps to improve response accuracy: generating a baseline response, planning verification questions, executing those verifications to check for inconsistencies, and generating a final verified response based on the results.


### Enhanced RAG Pipeline Features

In [84]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely. \nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is LLM Fine-tuning?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)

print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:1000])


Retrieving documents for query: 'What is LLM Fine-tuning?'
Top K: 3, Score threshold: 0.1 
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.49it/s]


Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Answer: LLM Fine-tuning is a targeted process of adjusting Large Language Models based on specific scenario and data characteristics to yield better results. It allows for the injection of additional domain knowledge when the model lacks it and enables the adjustment of the model’s input and output to adapt to specific data formats and response styles.
Sources: [{'source': '../data/pdf/RAG_doc.pdf', 'page': 9, 'score': 0.22952395677566528, 'preview': 'Extraction (IE) tasks. Another straightforward and effective\napproach involves having the LLM evaluate the retrieved\ncontent before generating the final answer. This allows the\nLLM to filter out documents with poor relevance through LLM\ncritique. For instance, in Chatlaw [104], the LLM is prompted\nt...'}]
Confidence: 0.22952395677566528
Context Preview: Extraction (IE) tasks. Another straightforward and effective
approach involves having the LLM evaluat

### Advanced RAG Pipeline: Streaming, Citations, History, Summarization

In [85]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is modular RAG", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is modular RAG'
Top K: 3, Score threshold: 0.1 
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.50it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
a key strategy. This concept has been implemented in frame-
works such as LlamaIndex2, LangChain3, and HayStack [12].
Feeding all relevant documents directly into LLMs can

 lead
to information overload, diluting the focus on key details with
irrelevant content.To mitigate this, post-retrieval efforts con-
centrate on selecting the essential information, emphasizing
critical sections, and shortening the context to be processed.
2https://www.llamaindex.ai
3https://www.langchain.com/
C. Modular RAG
The modular RAG architecture advances beyond the for-
mer two RAG paradigms, offering enhanced adaptability and
versatility. It incorporates diverse strategies for improving its
components, such as adding a search module for similarity
searches and refining the retriever through fine-tuning. Inno-
vations like restructured RAG modules [13] and rearranged
RAG pipelines [14] have been introduced to tackle specific
challenges. The shift towards a modular RAG approach is

to the user’s query. These articles, combined with the original
question, form a comprehensive prompt that empowers LLMs
to generate a well-informed answer.
The RAG research paradigm is continuously